In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT, ENTERPRISE_ATTACK_DATA, MOBILE_ATTACK_DATA, ICS_ATTACK_DATA
from apt_project import *

In [24]:
import json
import pandas as pd
from rapidfuzz import fuzz, process

Load the JSON and create a master DataFrame of STIX objects

In [25]:
enterprise_file = ENTERPRISE_ATTACK_DATA / "enterprise-attack.json"
mobile_file = MOBILE_ATTACK_DATA / "mobile-attack.json"
ics_file = ICS_ATTACK_DATA / "ics-attack.json"

with open(enterprise_file, "r", encoding="utf-8") as f:
    stix = json.load(f)

objects = stix.get("objects", [])
# Master dataframe (keeps the raw dict for reference)
master_df = pd.DataFrame([{
    "id": o.get("id"),  
    "type": o.get("type"),
    "name": o.get("name"),
    "description": o.get("description"),
    "created": o.get("created"),
    "modified": o.get("modified"),
    "raw": o
} for o in objects])

#master_df.head()


Extract techniques (ATT&CK `attack-pattern` objects)

In [26]:
tech_objs = [o for o in objects if o.get("type") == "attack-pattern"]

def technique_row(o):
    # external id e.g. T1003 usually in external_references where source_name == 'mitre-attack'
    ext_refs = o.get("external_references", [])
    mitre_ref = next((r for r in ext_refs if r.get("source_name") == "mitre-attack"), {})
    external_id = mitre_ref.get("external_id")
    # kill_chain_phases may contain tactic phase names
    kcp = o.get("kill_chain_phases") or o.get("kill_chain_phases", []) or []
    phases = [p.get("phase_name") for p in kcp if isinstance(p, dict) and p.get("phase_name")]
    platforms = o.get("x_mitre_platforms") or o.get("x-mitre-platforms") or []
    data_sources = o.get("x_mitre_data_sources") or []
    return {
        "id": o.get("id"),
        "tech_name": o.get("name"),
        "external_id": external_id,
        "description": o.get("description"),
        "platforms": platforms,
        "kill_chain_phases": phases,
        "raw": o
    }

tech_df = pd.DataFrame([technique_row(o) for o in tech_objs])
#tech_df.head()


Extract tactics (from kill_chain_phases) — canonicalize into a DataFrame

In [27]:
# Many ATT&CK bundles don't provide tactic objects as separate 'x-mitre-tactic' entries,
# but techniques include kill_chain_phases referencing 'mitre-attack' phase_name (tactic).
# We'll get the unique list of tactics from the techniques kill_chain_phases:

tactics = sorted({phase for phases in tech_df["kill_chain_phases"].tolist() for phase in phases if phase})
tactics_df = pd.DataFrame({"tactic": tactics})
#tactics_df


Relationships DataFrame (useful for group->technique and other links)

In [28]:
rel_objs = [o for o in objects if o.get("type") == "relationship"]

rel_df = pd.DataFrame([{
    "id": o.get("id"),
    "relationship_type": o.get("relationship_type"),
    "source_ref": o.get("source_ref"),
    "target_ref": o.get("target_ref"),
    "description": o.get("description"),
    "raw": o
} for o in rel_objs])

#rel_df.head()


Map techniques to tactics (exploded rows, easy to group)

In [29]:
# explode kill_chain_phases into row-per-technique-per-tactic
tech_exploded = tech_df.explode("kill_chain_phases").rename(columns={"kill_chain_phases":"tactic"})
tech_exploded = tech_exploded[["id","external_id","tech_name","tactic","platforms"]]
#tech_exploded.head()

## Create Dataframes

In [ ]:
# Filter intrusion-set objects (groups)
group_objs = [o for o in objects if o.get("type") == "intrusion-set"]

def group_row(o):
    return {
        "id": o.get("id"),
        "name": o.get("name"),
        "aliases": o.get("aliases", []),
        "description": o.get("description"),
        "created": o.get("created"),
        "modified": o.get("modified"),
        "raw": o
    }

groups_df = pd.DataFrame([group_row(g) for g in group_objs])

#groups_df

,id,name,aliases,description,created,modified,raw
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,"[Indrik Spider, Evil Corp, Manatee Tempest, DE...",[Indrik Spider](https://attack.mitre.org/group...,2021-01-06T17:46:35.134Z,2024-10-28T19:11:56.485Z,"{'modified': '2024-10-28T19:11:56.485Z', 'name..."
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,[LuminousMoth],[LuminousMoth](https://attack.mitre.org/groups...,2023-02-23T15:31:38.829Z,2025-04-16T20:37:32.806Z,"{'type': 'intrusion-set', 'spec_version': '2.1..."
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,Medusa Group,[Medusa Group],[Medusa Group](https://attack.mitre.org/groups...,2025-10-15T18:54:55.000Z,2025-10-24T04:01:48.210Z,"{'type': 'intrusion-set', 'spec_version': '2.1..."
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,"[Wizard Spider, UNC1878, TEMP.MixMaster, Grim ...",[Wizard Spider](https://attack.mitre.org/group...,2020-05-12T18:15:29.396Z,2025-03-12T20:33:21.597Z,"{'modified': '2025-03-12T20:33:21.597Z', 'name..."
4,intrusion-set--03506554-5f37-4f8f-9ce4-0e9f01a...,Elderwood,"[Elderwood, Elderwood Gang, Beijing Group, Sne...",[Elderwood](https://attack.mitre.org/groups/G0...,2018-04-18T17:59:24.739Z,2024-11-17T19:55:07.587Z,"{'modified': '2024-11-17T19:55:07.587Z', 'name..."
...,...,...,...,...,...,...,...
182,intrusion-set--59140a2e-d117-4206-9b2c-2a8662b...,Taidoor,[Taidoor],[Taidoor](https://attack.mitre.org/groups/G001...,2017-05-31T21:31:52.018Z,2025-04-18T17:59:28.548Z,"{'type': 'intrusion-set', 'spec_version': '2.1..."
183,intrusion-set--8332952e-b86b-486b-acc3-1c2a85d...,APT-C-23,"[APT-C-23, Mantis, Arid Viper, Desert Falcon, ...",[APT-C-23](https://attack.mitre.org/groups/G10...,2024-03-26T18:38:00.759Z,2024-11-17T20:01:55.806Z,"{'modified': '2024-11-17T20:01:55.806Z', 'name..."
184,intrusion-set--9559ecaf-2e75-48a7-aee8-9974020...,MONSOON,[],None,2017-05-31T21:32:07.928Z,2025-04-25T14:49:59.530Z,"{'type': 'intrusion-set', 'spec_version': '2.1..."
185,intrusion-set--92d5b3fd-3b39-438e-af68-770e447...,Charming Kitten,[Charming Kitten],[Charming Kitten](https://attack.mitre.org/gro...,2018-01-16T00:14:20.562Z,2025-04-18T17:59:28.881Z,"{'type': 'intrusion-set', 'spec_version': '2.1..."


In [31]:
# Filter relevant "uses" relationships
uses_rels = [
    r for r in rel_objs
    if r.get("relationship_type") == "uses"
    and r.get("source_ref", "").startswith("intrusion-set--")
    and r.get("target_ref", "").startswith("attack-pattern--")
]

rows = []
for r in uses_rels:
    group_id = r["source_ref"]
    tech_id = r["target_ref"]

    # Lookup group name
    group_name = next((g["name"] for g in group_objs if g["id"] == group_id), None)

    # Lookup technique information
    tech = next((t for t in tech_df.to_dict("records") if t["id"] == tech_id), None)

    rows.append({
        "group_id": group_id,
        "group_name": group_name,
        "technique_id": tech_id,
        "technique_name": tech.get("name") if tech else None,
        "tactic": tech.get("kill_chain_phases") if tech else None,
        "technique_description": tech.get("description") if tech else None,
        "relationship_description": r.get("description"),
    })

group_techniques_df = pd.DataFrame(rows)

In [33]:
tech_counts = (
    group_techniques_df
        .groupby("group_name")["technique_id"]
        .nunique()                      # count unique techniques per group
        .reset_index(name="technique_count")
        .sort_values("technique_count", ascending=False)
)

#tech_counts

## UMD Cyber Events Database

In [49]:
umd_df = pd.read_csv("umd_cyber_events_database.csv")

# Drop unnecessary columns
drop_cols = ["nato", "eu", "shanghai_coop", "oas", "mercosur", "au", "ecowas", "asean", "opec", "gulf_coop", "g7", "g20", "aukus", "csto", "oecd", "osce", "five_eyes"]

umd_df = umd_df.drop(columns=drop_cols)

#umd_df

In [50]:
# Prep list of all known APT groups and their aliases

def normalize(s):
    return str(s).strip().lower()

# 1. Canonical names
apt_names = groups_df["name"].dropna().tolist()

# 2. All alias lists flattened
alias_lists = groups_df["aliases"].dropna().tolist()
apt_aliases = [alias for sublist in alias_lists for alias in sublist]

# 3. Combined list
all_apt_terms = apt_names + apt_aliases

# 4. Normalize
all_apt_terms_norm = [normalize(x) for x in all_apt_terms]

# 5. Deduplicate + sort
all_apt_terms_norm = sorted(set(all_apt_terms_norm))

# 6. Build alias → canonical mapping
apt_map = {}
for _, row in groups_df.iterrows():
    canon = normalize(row["name"])
    apt_map[canon] = canon
    if isinstance(row["aliases"], list):
        for alias in row["aliases"]:
            apt_map[normalize(alias)] = canon

In [55]:
# Normalize event text
umd_df["actor_norm"] = umd_df["actor"].fillna("").str.lower()
umd_df["description_norm"] = umd_df["description"].fillna("").str.lower()

# Fuzzy match on actor column
def fuzzy_match_actor(actor):
    if not isinstance(actor, str) or not actor.strip():
        return None  # no match
    
    actor_norm = normalize(actor)
    result = process.extractOne(actor_norm, all_apt_terms_norm, scorer=fuzz.token_sort_ratio)

    if not result:
        return None
    
    match_term, score, idx = result

    if score >= 85:
        # return canonical group name
        return apt_map.get(match_term, None)
    
    return None

umd_df["actor_apt_match"] = umd_df["actor_norm"].apply(fuzzy_match_actor)

# Fuzzy match on description column
def fuzzy_match_description(text):
    if not isinstance(text, str) or not text.strip():
        return None
    
    words = normalize(text).split()

    for w in words:
        if len(w) < 3:
            continue
        
        result = process.extractOne(w, all_apt_terms_norm, scorer=fuzz.token_sort_ratio)
        if not result:
            continue
        
        match_term, score, idx = result
        
        if score >= 95:
            return apt_map.get(match_term, None)
    
    return None

umd_df["description_apt_match"] = umd_df["description_norm"].apply(fuzzy_match_description)

umd_df["apt_group"] = umd_df["actor_apt_match"].combine_first(umd_df["description_apt_match"])
#umd_df["apt_group"] = umd_df["actor_apt_match"]

# Create the final filtered dataframe
df_apts_only = umd_df[umd_df["apt_group"].notna()].reset_index(drop=True)

df_apts_only

,slug,original_method,event_date,reported_date,year,month,actor,actor_type,organization,industry_code,...,country,actor_country,state,county,change_log,actor_norm,description_norm,actor_apt_match,description_apt_match,apt_group
0,eac70b24ab5bd024,1,2022-06-15,NaN,2022,6,Blue Mockingbird,Criminal,Vulnerable Telerik UI systems,99,...,Undetermined,Korea (the Democratic People's Republic of),NaN,NaN,NaN,blue mockingbird,a threat actor known as 'blue mockingbird' is ...,blue mockingbird,None,blue mockingbird
1,0a8f23cbbf3107a2,1,2015-01-28,NaN,2015,1,Undetermined,Criminal,Hope FM,51,...,Kenya,Undetermined,NaN,NaN,NaN,undetermined,"suspected hackers briefly hijack hope fm, a po...",None,play,play
2,649f14c0fd38146e,1,2015-05-31,NaN,2015,5,APT32,Nation-State,Armed Forces of the Philippines,92,...,Philippines,Viet Nam,NaN,NaN,NaN,apt32,vietnamese threat actor ocean lotus has compro...,apt32,None,apt32
3,c786e72aab5694ab,1,2015-05-31,NaN,2015,5,APT32,Nation-State,ASEAN Investment,92,...,Undetermined,Viet Nam,NaN,NaN,NaN,apt32,vietnamese threat actor ocean lotus has compro...,apt32,None,apt32
4,7367f09a029020bf,1,2015-05-31,NaN,2015,5,APT32,Nation-State,ASEAN Trade Repository,92,...,Undetermined,Viet Nam,NaN,NaN,NaN,apt32,vietnamese threat actor ocean lotus has compro...,apt32,None,apt32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608,m4v2x9c7p1r5k8t3,1,2025-09-01,2025-10-22,2025,9,Ministry of Intelligence and Security (MOIS) (...,Nation-state,At least one undisclosed government entity in ...,92,...,Undetermined,Iran (Islamic Republic of),Undetermined,Undetermined,NaN,ministry of intelligence and security (mois) (...,reporting indicates a sustained espionage wave...,None,muddywater,muddywater
609,c8d1e6f3a0b9k7p4,1,2025-10-22,2025-10-22,2025,10,Coldriver,Nation-state,At least one undisclosed Ukraine war-relief or...,81,...,Ukraine,Russian Federation,NaN,NaN,NaN,coldriver,targeted credential-theft/implant delivery aga...,star blizzard,None,star blizzard
610,q6v2r9x1c7m5t3k8,1,2025-09-01,2025-10-24,2025,9,APT36 (Transparent Tribe),Nation-state,Undisclosed Indian government or infrastructur...,92,...,India,Pakistan,NaN,NaN,NaN,apt36 (transparent tribe),pakistan-linked apt36 used themed lures and ht...,None,transparent tribe,transparent tribe
611,b9e1f4d2c7a3e805,1,2025-09-01,2025-10-30,2025,9,UNC6384,Nation-State,Undisclosed Belgian Diplomatic Entities,92,...,Belgium,China,NaN,NaN,NaN,unc6384,china-linked unc6384 conducted a cyber-espiona...,mustang panda,mustang panda,mustang panda


## Export Dataframes

In [ ]:
def main_df():
    return master_df

def tech_df():
    return tech_df

def tactics_df():
    return tactics_df

def rel_df():
    return rel_df

def groups_df():
    return groups_df

def group_techniques_df():
    return group_techniques_df

: 